In [1]:
import pandas as pd

In [32]:
campo_tec = pd.read_csv('C:\\Users\\ECO\\Desktop\\ITI - Polarização\\Patentes\\badepiv11_ipc_campo_tec.csv', sep=';', encoding='latin-1')
depositante = pd.read_csv('C:\\Users\\ECO\\Desktop\\ITI - Polarização\\Patentes\\badepiv11_ptn_depositante.csv', sep=';', encoding='latin-1')
deposito = pd.read_csv('C:\\Users\\ECO\\Desktop\\ITI - Polarização\\Patentes\\badepiv11_ptn_deposito.csv', sep=';', encoding="utf-8")
despacho = pd.read_csv('C:\\Users\\ECO\\Desktop\\ITI - Polarização\\Patentes\\badepiv11_ptn_despacho.csv', sep=';', encoding='latin-1')
inventor = pd.read_csv('C:\\Users\\ECO\\Desktop\\ITI - Polarização\\Patentes\\badepiv11_ptn_inventor.csv', sep=';', encoding='latin-1')
ipc_campo_tec = pd.read_csv('C:\\Users\\ECO\\Desktop\\ITI - Polarização\\Patentes\\badepiv11_ptn_ipc_campo_tec.csv', sep=';', encoding='latin-1')
ptn_pct = pd.read_csv('C:\\Users\\ECO\\Desktop\\ITI - Polarização\\Patentes\\badepiv11_ptn_pct.csv', sep=';', encoding='latin-1')
prioridade = pd.read_csv('C:\\Users\\ECO\\Desktop\\ITI - Polarização\\Patentes\\badepiv11_ptn_prioridade.csv', sep=';', encoding='latin-1')

C:\Users\ECO\AppData\Local\Temp\ipykernel_28116\3673620170.py:1: DtypeWarning: Columns (0: NO_PEDIDO) have mixed types. Specify dtype option on import or set low_memory=False.
  campo_tec = pd.read_csv('C:\\Users\\ECO\\Desktop\\ITI - Polarização\\Patentes\\badepiv11_ipc_campo_tec.csv', sep=';', encoding='latin-1')
C:\Users\ECO\AppData\Local\Temp\ipykernel_28116\3673620170.py:3: DtypeWarning: Columns (0: NO_PEDIDO) have mixed types. Specify dtype option on import or set low_memory=False.
  deposito = pd.read_csv('C:\\Users\\ECO\\Desktop\\ITI - Polarização\\Patentes\\badepiv11_ptn_deposito.csv', sep=';', encoding="utf-8")
C:\Users\ECO\AppData\Local\Temp\ipykernel_28116\3673620170.py:4: DtypeWarning: Columns (0: NO_PEDIDO) have mixed types. Specify dtype option on import or set low_memory=False.
  despacho = pd.read_csv('C:\\Users\\ECO\\Desktop\\ITI - Polarização\\Patentes\\badepiv11_ptn_despacho.csv', sep=';', encoding='latin-1')
C:\Users\ECO\AppData\Local\Temp\ipykernel_28116\3673620170

In [ ]:
df_geral = pd.read_csv('')

,NO_PEDIDO,NO_ORDEM_CLASSE,CD_CLASSIF,VERSION_INDICATOR,CAMPO_TEC
0,102012000050,1,H04W 4/12,2009.01,4.0
1,102012000050,2,B60L 11/18,1968.09,32.0
2,102012000050,3,H02J 7/00,2006.01,1.0
3,102012000052,1,H01R 13/627,2006.01,1.0
4,102012000052,2,H01R 27/00,2006.01,1.0


In [ ]:
import numpy as np
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import os

# 1. Inicializar o modelo e referências globalmente para uso na função
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')



def classificar_patentes(df_in, coluna='abstract', batch_size=128):
    textos = df_in[coluna].fillna('').astype(str).tolist()

    # Ordenar por tamanho acelera, pois reduz o padding dentro de cada lote
    ordem = np.argsort([len(t) for t in textos])
    textos_ord = [textos[i] for i in ordem]

    embs = []
    for i in tqdm(range(0, len(textos_ord), batch_size),
                  desc="Codificando patentes", unit="lote"):
        lote = textos_ord[i:i + batch_size]
        emb = model.encode(
            lote,
            batch_size=batch_size,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )
        embs.append(emb)

    emb = torch.cat(embs)

    # Similaridade vetorizada
    sims_ord = (emb @ emb_ref.to(emb.device).T).float().cpu().numpy()

    # Restaura a ordem original
    sims = np.empty_like(sims_ord)
    sims[ordem] = sims_ord

    out = df_in.copy()
    out['sim_processo'] = sims[:, 0]
    out['sim_produto']  = sims[:, 1]
    out['classificacao'] = np.where(
        out['sim_processo'] > out['sim_produto'],
        'Inovação de Processo',
        'Inovação de Produto'
    )
    return out

    

# Carregar as definições de processo e produto para criar a matriz de referência
def_processo = open('/content/drive/MyDrive/Patentes/Processo 10, 11 e 12.csv', 'r', encoding='utf-8').read().strip()
def_produto = open('/content/drive/MyDrive/Patentes/Produto 10, 11 e 12.csv', 'r', encoding='utf-8').read().strip()

# Gerar os embeddings de referência (dimensão: [2, embedding_size])
emb_ref = torch.tensor(model.encode([def_processo, def_produto]), dtype=torch.float32)

resultado = classificar_patentes(setor10)
resultado[['abstract', 'sim_processo', 'sim_produto', 'classificacao']].head()

,NO_PEDIDO,NO_ORDEM,NO_CNPJ_CPF,CD_TIPO_PFPJ,NM_COMPLET_PFPJ,CD_PAIS_PFPJ,CD_UF_PFPJ,NM_CIDADE_PFPJ,CD_IBGE_CIDADE
0,PI0311748,1,NaN,NaN,The Coca-Cola Company,US,NaN,Atlanta,0
1,MU8401405,1,61064978000101,J,Arno S/A,BR,SP,São Paulo,3550308
2,MU8402592,1,61064978000101,J,Arno S/A,BR,SP,São Paulo,3550308
3,PI0309821,1,NaN,NaN,THE GILLETTE COMPANY,US,NaN,NaN,0
4,PI0407779,1,NaN,NaN,THE GILLETTE COMPANY,US,NaN,NaN,0


In [ ]:
import os, re
import numpy as np
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

PASTA = 'C:\\Users\\ECO\\Desktop\\ITI - Polarização\\Patentes\\Definição_Inovações_Pintec_2017'
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')


def ler_texto(caminho):
    with open(caminho, encoding='utf-8') as f:
        return f.read().strip()

# 1. Descobre os pares Processo/Produto e a quais setores cada um se aplica
def carregar_referencias(pasta):
    arquivos = {'Processo': {}, 'Produto': {}}
    for nome in os.listdir(pasta):
        m = re.match(r'^(Processo|Produto)[ _](.+)\.(csv|txt)$', nome)
        if m:
            arquivos[m.group(1)][m.group(2).strip()] = os.path.join(pasta, nome)

    grupos, setor_para_grupo, idx_geral = [], {}, None
    for chave, caminho_proc in arquivos['Processo'].items():
        if chave not in arquivos['Produto']:
            print(f"⚠️ Sem par de Produto para '{chave}', ignorado.")
            continue
        textos = [ler_texto(caminho_proc), ler_texto(arquivos['Produto'][chave])]
        emb_ref = model.encode(textos, convert_to_tensor=True, normalize_embeddings=True)

        grupos.append({'nome': chave, 'emb_ref': emb_ref})
        i = len(grupos) - 1
        if chave.lower() == 'geral':
            idx_geral = i
        else:
            for s in re.findall(r'\d+', chave):
                setor_para_grupo[int(s)] = i
    return grupos, setor_para_grupo, idx_geral

grupos, setor_para_grupo, idx_geral = carregar_referencias(PASTA)
print(f"{len(grupos)} grupos carregados; setores mapeados: {sorted(setor_para_grupo)}")


# 2. Classifica qualquer DataFrame que tenha a coluna ISIC e o abstract
def classificar_patentes(df_in, coluna_texto='abstract', coluna_setor='ISIC', batch_size=128):
    df_in = df_in.reset_index(drop=True)
    textos = df_in[coluna_texto].fillna('').astype(str).tolist()

    # Codifica todos os abstracts UMA vez (ordenados por tamanho, com barra de progresso)
    ordem = np.argsort([len(t) for t in textos])
    textos_ord = [textos[i] for i in ordem]
    embs = []
    for i in tqdm(range(0, len(textos_ord), batch_size), desc="Codificando", unit="lote"):
        embs.append(model.encode(textos_ord[i:i + batch_size], batch_size=batch_size,
                                 convert_to_tensor=True, normalize_embeddings=True,
                                 show_progress_bar=False))
    emb = torch.cat(embs)[torch.as_tensor(np.argsort(ordem), device=embs[0].device)]

    # Descobre o grupo de referência de cada linha (setor sem arquivo -> Geral)
    setores = pd.to_numeric(df_in[coluna_setor], errors='coerce')
    idx_grupo = np.array([
        setor_para_grupo.get(int(s), idx_geral) if pd.notna(s) else idx_geral
        for s in setores
    ])
    if (idx_grupo == None).any():
        raise ValueError("Há setores sem grupo e não existe arquivo 'Geral'.")
    idx_grupo = idx_grupo.astype(int)

    # Similaridade vetorizada, um grupo por vez
    sim = np.zeros((len(df_in), 2), dtype=np.float32)
    for g in np.unique(idx_grupo):
        mask = idx_grupo == g
        sim[mask] = (emb[torch.as_tensor(mask, device=emb.device)] @ grupos[g]['emb_ref'].T).float().cpu().numpy()

    out = df_in.copy()
    out['sim_processo'] = sim[:, 0]
    out['sim_produto'] = sim[:, 1]
    out['diferenca'] = (sim[:, 0] - sim[:, 1]).round(4)
    out['classificacao'] = np.where(sim[:, 0] > sim[:, 1], 'Inovação de Processo', 'Inovação de Produto')
    out['grupo_referencia'] = [grupos[g]['nome'] for g in idx_grupo]
    return out

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 663.04it/s]


7 grupos carregados; setores mapeados: [10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 58, 171, 172, 173, 174, 592]


In [4]:
df = pd.read_csv('df_automacao_limpo_ISIC.csv')
df.head()


,Unnamed: 0,publication_number,filing_date,ipc_codes,title,abstract,4-digitos IPC,ISIC
0,0,BR-112020007064-A2,20180620,D04H,folha elástica para artigo absorvente e artigo...,A presente invenção refere-se a folha elástica...,D04H,13
1,1,BR-112013006594-B1,20110920,C07D,Composto ciclopropano antagonista de receptor ...,"COMPOSTO, COMPOSIÇÃO FARMACÊUTICA, MÉTODO DE T...",C07D,21
2,2,BR-112021026164-A2,20200625,G07D,Máquina de transação automatizada e cassete pa...,MÁQUINA DE TRANSAÇÃO AUTOMATIZADA E CASSETE PA...,G07D,32
3,3,BR-112018070089-A2,20170331,H04L,métodos de operação de um dispositivo sem fio ...,são providos sistemas e métodos para relato da...,H04L,26
4,4,BR-202021008981-U2,20210507,A22C,Equipamento automático para montar espetinhos ...,EQUIPAMENTO AUTOMÁTICO PARA MONTAR ESPETINHOS ...,A22C,10


In [ ]:
# Todos os setores de uma vez (usa o ISIC de cada linha para escolher o par certo)
# resultado = classificar_patentes(df)

# Ou apenas um DataFrame específico
# Filtra o setor 23 garantindo que o ISIC seja tratado como número
df_23 = df[pd.to_numeric(df['ISIC'], errors='coerce').eq(23)].copy()

if df_23.empty:
    raise ValueError("Nenhuma patente encontrada para o setor ISIC 23.")

resultado = classificar_patentes(df_23)

resultado[['ISIC', 'grupo_referencia', 'classificacao', 'diferenca']].head()
resultado['classificacao'].value_counts()

Codificando: 100%|██████████| 80/80 [13:13<00:00,  9.92s/lote]


classificacao
Inovação de Produto     9165
Inovação de Processo    1047
Name: count, dtype: int64

In [ ]:
resultado.info()

<class 'pandas.DataFrame'>
RangeIndex: 10212 entries, 0 to 10211
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          10212 non-null  int64  
 1   publication_number  10212 non-null  str    
 2   filing_date         10212 non-null  int64  
 3   ipc_codes           10212 non-null  str    
 4   title               10212 non-null  str    
 5   abstract            10212 non-null  str    
 6   4-digitos IPC       10212 non-null  str    
 7   ISIC                10212 non-null  str    
 8   sim_processo        10212 non-null  float32
 9   sim_produto         10212 non-null  float32
 10  diferenca           10212 non-null  float32
 11  classificacao       10212 non-null  str    
 12  grupo_referencia    10212 non-null  str    
dtypes: float32(3), int64(2), str(8)
memory usage: 11.0 MB


In [19]:
import os, hashlib
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

# ---------------- CONFIGURAÇÃO ----------------
PASTA_REFS  = r'C:\Users\ECO\Desktop\ITI - Polarização\Patentes\Definição_Inovações_Pintec_2017'
PASTA_CACHE = r'C:\Users\ECO\Desktop\ITI - Polarização\Patentes\cache_embeddings'
# Colab: PASTA_CACHE = '/content/drive/MyDrive/Patentes/cache_embeddings'

NOME_MODELO = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
MAX_SEQ     = 128     # tokens por texto (aumentar deixa mais lento)
BATCH_SIZE  = 64      # diminua se faltar memória
BLOCO       = 2000    # a cada quantos textos salva um checkpoint
# -----------------------------------------------

os.makedirs(PASTA_CACHE, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo: {device}")   # se aparecer 'cpu', vai ser lento

model = SentenceTransformer(NOME_MODELO, device=device)
model.max_seq_length = MAX_SEQ


# ============ 1. REFERÊNCIAS (pasta de setores) ============
import re

def ler_texto(caminho):
    with open(caminho, encoding='utf-8-sig') as f:   # aceita UTF-8 com ou sem BOM
        return f.read().strip()

def carregar_referencias(pasta):
    arquivos = {'Processo': {}, 'Produto': {}}
    for nome in os.listdir(pasta):
        m = re.match(r'^(Processo|Produto)[ _](.+)\.(csv|txt)$', nome)
        if m:
            arquivos[m.group(1)][m.group(2).strip()] = os.path.join(pasta, nome)

    grupos, setor_para_grupo, idx_geral = [], {}, None
    for chave, caminho_proc in arquivos['Processo'].items():
        if chave not in arquivos['Produto']:
            print(f"⚠️ Sem par de Produto para '{chave}', ignorado.")
            continue
        textos = [ler_texto(caminho_proc), ler_texto(arquivos['Produto'][chave])]
        emb_ref = model.encode(textos, convert_to_numpy=True, normalize_embeddings=True)
        grupos.append({'nome': chave, 'emb_ref': emb_ref.astype(np.float32)})
        i = len(grupos) - 1
        if chave.lower() == 'geral':
            idx_geral = i
        else:
            for s in re.findall(r'\d+', chave):
                setor_para_grupo[int(s)] = i
    return grupos, setor_para_grupo, idx_geral

grupos, setor_para_grupo, idx_geral = carregar_referencias(PASTA_REFS)
print(f"{len(grupos)} grupos: {[g['nome'] for g in grupos]}")
print(f"Setores mapeados: {sorted(setor_para_grupo)}")
print(f"Grupo 'Geral' encontrado: {idx_geral is not None}")


# ============ 2. VALIDAÇÃO ANTES DE GASTAR HORAS ============
def validar(df_in, coluna_setor='ISIC', coluna_texto='abstract'):
    isic = pd.to_numeric(df_in[coluna_setor], errors='coerce')
    sem_grupo = ~isic.isin(setor_para_grupo.keys())
    print(f"Linhas: {len(df_in)} | sem ISIC válido ou sem grupo próprio: {sem_grupo.sum()}")
    if sem_grupo.any():
        print(isic[sem_grupo].value_counts(dropna=False).head(15))
        if idx_geral is None:
            print("⚠️ Não há arquivo 'Geral': essas linhas ficarão como 'Sem referência'.")
    vazios = df_in[coluna_texto].fillna('').astype(str).str.strip().eq('').sum()
    print(f"Abstracts vazios: {vazios}")

validar(df)


# ============ 3. CODIFICAÇÃO COM CHECKPOINT ============
def codificar_com_checkpoint(textos):
    # Impressão digital: se os textos ou o modelo mudarem, usa outra subpasta
    h = hashlib.md5()
    h.update(f"{NOME_MODELO}|{MAX_SEQ}|{len(textos)}".encode())
    for t in textos:
        h.update(t.encode('utf-8', errors='ignore'))
    pasta = os.path.join(PASTA_CACHE, h.hexdigest()[:12])
    os.makedirs(pasta, exist_ok=True)

    # Ordena por tamanho (menos padding) e guarda a ordem
    ordem = np.argsort([len(t) for t in textos], kind='stable')
    np.save(os.path.join(pasta, 'ordem.npy'), ordem)
    textos_ord = [textos[i] for i in ordem]

    n_blocos = (len(textos_ord) + BLOCO - 1) // BLOCO
    for b in tqdm(range(n_blocos), desc="Blocos", unit="bloco"):
        caminho = os.path.join(pasta, f'bloco_{b:05d}.npy')
        if os.path.exists(caminho):
            continue                                  # já feito: pula
        trecho = textos_ord[b * BLOCO:(b + 1) * BLOCO]
        emb = model.encode(trecho, batch_size=BATCH_SIZE, convert_to_numpy=True,
                           normalize_embeddings=True, show_progress_bar=False)
        tmp = caminho + '.tmp.npy'
        np.save(tmp, emb.astype(np.float32))
        os.replace(tmp, caminho)                      # gravação "atômica": sem arquivo corrompido

    emb_ord = np.concatenate(
        [np.load(os.path.join(pasta, f'bloco_{b:05d}.npy')) for b in range(n_blocos)])
    emb = np.empty_like(emb_ord)
    emb[ordem] = emb_ord                              # volta à ordem original
    np.save(os.path.join(pasta, 'embeddings_final.npy'), emb)
    print(f"Embeddings salvos em: {pasta}")
    return emb


# ============ 4. CLASSIFICAÇÃO (rápida: só comparações) ============
def classificar_patentes(df_in, emb, coluna_setor='ISIC', coluna_texto='abstract'):
    df_in = df_in.reset_index(drop=True)
    setores = pd.to_numeric(df_in[coluna_setor], errors='coerce')

    padrao = -1 if idx_geral is None else idx_geral
    idx_grupo = np.array([
        setor_para_grupo.get(int(s), padrao) if pd.notna(s) else padrao
        for s in setores
    ], dtype=int)

    sim = np.full((len(df_in), 2), np.nan, dtype=np.float32)
    for g in np.unique(idx_grupo):
        if g == -1:
            continue
        mask = idx_grupo == g
        sim[mask] = emb[mask] @ grupos[g]['emb_ref'].T     # cosseno (vetores normalizados)

    sem_ref = idx_grupo == -1
    vazio = df_in[coluna_texto].fillna('').astype(str).str.strip().eq('').values

    out = df_in.copy()
    out['sim_processo'] = sim[:, 0]
    out['sim_produto']  = sim[:, 1]
    out['diferenca']    = (sim[:, 0] - sim[:, 1]).round(4)
    out['classificacao'] = np.where(
        sem_ref, 'Sem referência',
        np.where(vazio, 'Abstract vazio',
                 np.where(sim[:, 0] > sim[:, 1], 'Inovação de Processo', 'Inovação de Produto')))
    out['grupo_referencia'] = [grupos[g]['nome'] if g != -1 else None for g in idx_grupo]
    return out


# ============ 5. EXECUÇÃO ============
# 5a. TESTE RÁPIDO (recomendado antes de rodar tudo)
amostra = df.head(200).reset_index(drop=True)
emb_teste = codificar_com_checkpoint(amostra['abstract'].fillna('').astype(str).tolist())
print(classificar_patentes(amostra, emb_teste)['classificacao'].value_counts())

# 5b. RODADA COMPLETA (pode interromper e rodar de novo: continua de onde parou)
textos = df['abstract'].fillna('').astype(str).tolist()
emb = codificar_com_checkpoint(textos)

resultado = classificar_patentes(df, emb)
print(resultado['classificacao'].value_counts())

# 5c. Salva o resultado final também
resultado.to_csv(os.path.join(PASTA_CACHE, 'resultado_classificado.csv'),
                 index=False, sep=';', encoding='utf-8-sig')

Dispositivo: cpu


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 610.18it/s]


8 grupos: ['10, 11 e 12', '13, 171, 172, 18, 58, 592', '14, 15, 16, 173, 174, 31, 32', '19, 20, 21 e 22', '23, 24', '25, 28, 29, 30', '26, 27', 'Geral']
Setores mapeados: [10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 58, 171, 172, 173, 174, 592]
Grupo 'Geral' encontrado: True
Linhas: 384327 | sem ISIC válido ou sem grupo próprio: 83822
ISIC
36.0    19041
59.0    17105
86.0     8621
9.0      7964
17.0     7292
46.0     3147
1.0      3096
35.0     3067
NaN      2724
3.0      2047
2.0      1742
43.0     1735
96.0     1356
92.0     1232
93.0     1078
Name: count, dtype: int64
Abstracts vazios: 0


Blocos: 100%|██████████| 1/1 [00:07<00:00,  7.83s/bloco]


Embeddings salvos em: C:\Users\ECO\Desktop\ITI - Polarização\Patentes\cache_embeddings\7e14a7ec4a1a
classificacao
Inovação de Produto     124
Inovação de Processo     76
Name: count, dtype: int64


Blocos: 100%|██████████| 193/193 [10:40:50<00:00, 199.23s/bloco] 


Embeddings salvos em: C:\Users\ECO\Desktop\ITI - Polarização\Patentes\cache_embeddings\34fac152169f
classificacao
Inovação de Produto     230996
Inovação de Processo    153331
Name: count, dtype: int64


In [20]:
resultado.to_csv('resultado_classificacao_patentes.csv', index=False, encoding='utf-8-sig')

In [21]:
resultado.info()

<class 'pandas.DataFrame'>
RangeIndex: 384327 entries, 0 to 384326
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Unnamed: 0          384327 non-null  int64  
 1   publication_number  384327 non-null  str    
 2   filing_date         384327 non-null  int64  
 3   ipc_codes           384327 non-null  str    
 4   title               384327 non-null  str    
 5   abstract            384327 non-null  str    
 6   4-digitos IPC       384327 non-null  str    
 7   ISIC                384327 non-null  str    
 8   sim_processo        384327 non-null  float32
 9   sim_produto         384327 non-null  float32
 10  diferenca           384327 non-null  float32
 11  classificacao       384327 non-null  str    
 12  grupo_referencia    384327 non-null  str    
dtypes: float32(3), int64(2), str(8)
memory usage: 407.2 MB


In [22]:
resultado.head()

,Unnamed: 0,publication_number,filing_date,ipc_codes,title,abstract,4-digitos IPC,ISIC,sim_processo,sim_produto,diferenca,classificacao,grupo_referencia
0,0,BR-112020007064-A2,20180620,D04H,folha elástica para artigo absorvente e artigo...,A presente invenção refere-se a folha elástica...,D04H,13,0.490508,0.585174,-0.0947,Inovação de Produto,"13, 171, 172, 18, 58, 592"
1,1,BR-112013006594-B1,20110920,C07D,Composto ciclopropano antagonista de receptor ...,"COMPOSTO, COMPOSIÇÃO FARMACÊUTICA, MÉTODO DE T...",C07D,21,0.179574,0.341748,-0.1622,Inovação de Produto,"19, 20, 21 e 22"
2,2,BR-112021026164-A2,20200625,G07D,Máquina de transação automatizada e cassete pa...,MÁQUINA DE TRANSAÇÃO AUTOMATIZADA E CASSETE PA...,G07D,32,0.318940,0.204251,0.1147,Inovação de Processo,"14, 15, 16, 173, 174, 31, 32"
3,3,BR-112018070089-A2,20170331,H04L,métodos de operação de um dispositivo sem fio ...,são providos sistemas e métodos para relato da...,H04L,26,0.369450,0.273137,0.0963,Inovação de Processo,"26, 27"
4,4,BR-202021008981-U2,20210507,A22C,Equipamento automático para montar espetinhos ...,EQUIPAMENTO AUTOMÁTICO PARA MONTAR ESPETINHOS ...,A22C,10,0.490840,0.377578,0.1133,Inovação de Processo,"10, 11 e 12"


In [ ]:
primeiras100 = resultado.head(100)
primeiras100.to_csv('primeiras100_patentes.csv', index=False, encoding='utf-8-sig')